# Results viewer — our models vs. the BIG-TB baselines

Two families of model live in `results/experiments/`:

| | what it is | how many models | baseline it belongs against |
|---|---|---|---|
| **single-drug** | `run_experiment.py` — one model per drug, one folder per modality config (`dna`, `dna_protein`, …) | 11 per config | **SD-CNN (OHE)** |
| **joint / multi-drug** | `run_multidrug.py` — ONE `MultiDrugNet` predicting all 11 drugs (`multidrug_*` folders) | 1 | **MD-CNN (OHE)** |

**Metric to judge on: cross-validated AUC (5-fold mean ± SD).** Both BIG-TB
models and both of our pipelines compute it the same way, and it averages five
splits instead of trusting one. Held-out test AUC is a *single* 20% split of one
best-CV-fold model — reported here because it is what the paper publishes, but it
swings by ±0.05 on the small drugs. `—` in a table is a run that hasn't finished,
never a zero.

**The SD-CNN baseline is leak-corrected.** The published SD-CNN test AUCs come
from an `assess` script that re-splits with the same seed but a different
`stratify` argument than the script that trained the model, so ~80% of its
"test" isolates were in training. Table 1 below quantifies it. The **MD-CNN**
baseline is *not* affected — its `assess` splits on a stored cohort column and
never re-splits — so its published numbers are used as-is.

**Layout:** setup → baseline correction → the two master tables → single-drug
figures → multi-drug section → headline numbers.

In [ ]:
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.transforms import blended_transform_factory
from IPython.display import display

In [ ]:
# --- config ---
# This notebook lives in <project>/notebooks/; find the project holding results.
_candidates = [
    Path("/home/jacksonmicha_umass_edu/abr_workspace/biophysical-fusion"),
    Path.cwd().parent,
    Path.cwd(),
]
PROJECT = next(
    (p for p in _candidates if (p / "results/experiments").is_dir()),
    _candidates[0],
)
EXP_DIR = PROJECT / "results/experiments"
print("Reading experiments from:", EXP_DIR)

# Readable label + display order per experiment folder. Folders found on disk
# but not listed here still load (labeled by folder name), appended at the end.
EXP_LABELS = {
    "dna":                 "DNA",
    "dna_protein":         "DNA + protein",
    "dna_biophysical":     "DNA + biophysical",
    "dna_regulatory":      "DNA + regulatory",
    "all_modalities":      "All modalities",
}
REFERENCE_EXP = "dna"   # multimodal gains are measured against this run

found = sorted(p.name for p in EXP_DIR.iterdir()
               if (p / "summary.csv").exists())
EXPERIMENTS = [e for e in EXP_LABELS if e in found] + \
              [e for e in found if e not in EXP_LABELS]
EXPERIMENTS = [e for e in EXPERIMENTS if "permod" not in e]  # ignore per-modality ablations
LABEL = {e: EXP_LABELS.get(e, e) for e in EXPERIMENTS}
print(f"{len(EXPERIMENTS)} experiments:", ", ".join(EXPERIMENTS))
_unlabeled = [e for e in EXPERIMENTS if e not in EXP_LABELS]
if _unlabeled:
    print("  (no readable label, shown by folder name):", ", ".join(_unlabeled))

# BIG-TB 1D CNN baseline as PUBLISHED (Tasmin et al., Table 4 — Test AUC).
# NOTE: the SD-CNN "test AUC" here is the reference authors' `assess` output,
# which is INFLATED by a train/test leak (see SDCNN_CLEAN below). Kept only as a
# reference; comparisons use the leak-corrected numbers.
#                     drug : (SD-CNN_published, MD-CNN_published)
BIGTB_1DCNN = {
    "AMIKACIN":     (0.845, 0.899),
    "CAPREOMYCIN":  (0.873, 0.829),
    "ETHAMBUTOL":   (0.931, 0.936),
    "ETHIONAMIDE":  (0.670, 0.709),
    "ISONIAZID":    (0.917, 0.951),
    "KANAMYCIN":    (0.849, 0.875),
    "LEVOFLOXACIN": (0.839, 0.850),
    "MOXIFLOXACIN": (0.886, 0.887),
    "PYRAZINAMIDE": (0.930, 0.908),
    "RIFAMPICIN":   (0.977, 0.981),
    "STREPTOMYCIN": (0.911, 0.938),
}

# --- Leak-corrected SD-CNN baseline (H1 finding, 2026-07-28) -----------------
# The published SD-CNN test AUC is inflated by a train/test leak: their `assess`
# script re-splits with the same seed but `stratify=y` — a DIFFERENT partition
# from the `crossval` script that TRAINED the saved model — so ~80% of the
# reported "test" isolates were in the model's TRAIN set. Re-running THEIR saved
# best model on the truly held-out crossval split (leak-free) gives the clean
# numbers below (verified: their published number reproduced EXACTLY for 10/11
# drugs; abr_workspace/h1_repro/eval_leak_all.py -> leak_all.csv). The leak
# materially inflates only the hard imbalanced drugs (MOXI/CAPREO/ETO); for the
# rest published ≈ clean.   drug : (clean_test_AUC, clean_CV_mean, published_leaky)
SDCNN_CLEAN = {
    "AMIKACIN":     (0.885, 0.859, 0.885),
    "CAPREOMYCIN":  (0.815, 0.847, 0.873),
    "ETHAMBUTOL":   (0.925, 0.926, 0.931),
    "ETHIONAMIDE":  (0.644, 0.622, 0.670),
    "ISONIAZID":    (0.917, 0.912, 0.917),
    "KANAMYCIN":    (0.855, 0.867, 0.849),
    "LEVOFLOXACIN": (0.885, 0.850, 0.839),
    "MOXIFLOXACIN": (0.825, 0.819, 0.886),
    "PYRAZINAMIDE": (0.922, 0.913, 0.930),
    "RIFAMPICIN":   (0.980, 0.972, 0.977),
    "STREPTOMYCIN": (0.924, 0.913, 0.911),
}
# Fold-to-fold spread of the SD-CNN's own 5-fold CV (Tasmin et al., Table 14).
# Means in that table match SDCNN_CLEAN's `clean CV` to 3 dp for 10/11 drugs, so
# the SDs pair with those values. ETHIONAMIDE is deliberately absent: its Table-14
# row (mean 0.865) contradicts the authors' auc.csv (0.622), so no SD is trusted
# for it -- see MDCNN_CV_SUSPECT below.
SDCNN_CV_SD = {
    "AMIKACIN":     0.0195,
    "CAPREOMYCIN":  0.0242,
    "ETHAMBUTOL":   0.0032,
    "ISONIAZID":    0.0119,
    "KANAMYCIN":    0.0294,
    "LEVOFLOXACIN": 0.0661,
    "MOXIFLOXACIN": 0.0272,
    "PYRAZINAMIDE": 0.0154,
    "RIFAMPICIN":   0.0014,
    "STREPTOMYCIN": 0.0120,
}

# Comparisons use the leak-corrected clean held-out AUC; set False for published.
USE_CLEAN_BASELINE = True
BASELINE_COL = "SD-CNN"   # kept for label continuity ("BIG-TB SD-CNN" column)

# --- BIG-TB MD-CNN: the MULTI-DRUG baseline --------------------------------
# The apples-to-apples target for our joint model (run_multidrug.py): ONE CNN
# predicting all 11 drugs at once from one-hot DNA over the union of loci.
#   test AUC : Tasmin et al., Table 4, column "MD-CNN (OHE)".
#   CV  AUC  : Tasmin et al., Table 14 (per-drug 5-fold mean +/- SD, DNA models).
#
# Protocol, read from Big-TB-benchmark/dna-tasks/MDCNN/model_training/:
#  * main_mdcnn_crossval.py -> train_test_split(test_size=0.2, random_state=42)
#    with NO stratify, then KFold(5, shuffle) inside the train split. That is
#    the SAME protocol as our run_multidrug_cv (non-stratified 80/20 seed 42 +
#    KFold-5), so CV-vs-CV is the matched comparison. Leak-free.
#  * main_mdcnn_assess.py -> does NOT re-split with a seed at all: it trains on
#    the predefined cohort `category == "set1_original_10202"` and tests on
#    every other isolate. The SD-CNN crossval/assess stratify LEAK THEREFORE
#    DOES NOT APPLY to MD-CNN. But their test cohort is a different (larger,
#    differently composed) set from our random 20% hold-out, so test-vs-test is
#    indicative only -- judge on CV.
#                 drug : (test_AUC_published, cv_mean, cv_std)
BIGTB_MDCNN = {
    "AMIKACIN":     (0.899, 0.9176, 0.0171),
    "CAPREOMYCIN":  (0.829, 0.8599, 0.0147),
    "ETHAMBUTOL":   (0.936, 0.9253, 0.0158),
    "ETHIONAMIDE":  (0.709, 0.9161, 0.0101),
    "ISONIAZID":    (0.951, 0.9708, 0.0034),
    "KANAMYCIN":    (0.875, 0.8925, 0.0183),
    "LEVOFLOXACIN": (0.850, 0.9450, 0.0072),
    "MOXIFLOXACIN": (0.887, 0.9298, 0.0135),
    "PYRAZINAMIDE": (0.908, 0.9113, 0.0146),
    "RIFAMPICIN":   (0.981, 0.9769, 0.0039),
    "STREPTOMYCIN": (0.938, 0.9276, 0.0079),
}

# Table 14's SD-CNN CV column reproduces the authors' own per-drug auc.csv to
# 3 dp for 10/11 drugs (= the `clean CV` values in SDCNN_CLEAN). ETHIONAMIDE is
# the exception: Table 14 says SD-CNN CV 0.865 / MD-CNN CV 0.916 while their
# auc.csv gives 0.622 and Table 4's test AUCs are 0.670 / 0.709. Something is
# off in that row, so its CV baselines are flagged, not trusted.
MDCNN_CV_SUSPECT = {"ETHIONAMIDE"}

# OUR multi-drug run folder -> readable label. Deliberately prefixed "OURS" so
# our joint model is never confused with the BIG-TB MD-CNN baseline it is
# compared against (both are "a multi-drug CNN"; only one of them is ours).
MD_LABELS = {
    "multidrug_dna_all":         "OURS: joint (DNA)",
    "multidrug_all_modalities":  "OURS: joint (all mods)",
    "multidrug_dna_regulatory":  "OURS: joint (DNA+reg)",
}
# ...and the SINGLE-drug experiment (from EXP_LABELS) that uses the SAME inputs,
# so "1 joint model vs 11 separate models" is an inputs-matched comparison.
MD_SD_MATCH = {
    "multidrug_dna_all":         "DNA",
    "multidrug_all_modalities":  "All modalities",
    "multidrug_dna_regulatory":  "DNA + regulatory",
}

# --- shared figure style (every plot below uses these) ----------------------
OUTDIR = PROJECT / "results/figures"        # set to None to display without saving
BLUE, ORANGE, VERM, GREY, INK = "#0072B2", "#E69F00", "#D55E00", "#7F7F7F", "#1A1A1A"
GOOD_BG, BAD_BG = "background-color:#d9ecd9;color:#14501e", "background-color:#f7d9d3;color:#7f2704"
RC = {
    "figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 12, "axes.titlesize": 13, "axes.labelsize": 12,
    "axes.titleweight": "bold", "axes.edgecolor": "0.3",
    "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": "0.25", "ytick.color": "0.25",
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
}
TABLE_STYLES = [
    {"selector": "th.col_heading", "props":
     "text-align:center; white-space:pre-line; font-size:12px; vertical-align:bottom;"},
    {"selector": "th.col_heading.level0", "props":
     "background-color:#eef2f6; border-bottom:2px solid #97a7b5;"},
    {"selector": "caption", "props":
     "caption-side:top; text-align:left; font-size:13px; padding-bottom:6px;"},
]


def save_fig(fig, name):
    if OUTDIR is None:
        return
    Path(OUTDIR).mkdir(parents=True, exist_ok=True)
    for ext in ("png", "pdf"):
        fig.savefig(Path(OUTDIR) / f"{name}.{ext}")


def colour_delta(col):
    """Styler helper: green where we're ahead, red where we trail."""
    return [GOOD_BG if v > 0.005 else (BAD_BG if v < -0.005 else "") for v in col]


def pm(mean, sd):
    """'0.912 ± 0.008' / '0.912' / '—'."""
    if pd.isna(mean):
        return "—"
    return f"{mean:.3f} ± {sd:.3f}" if not pd.isna(sd) else f"{mean:.3f}"

In [ ]:
# === LOAD ==================================================================
# Everything the rest of the notebook uses is built here, once:
#   cv_mean / cv_std  per-drug 5-fold CV AUC of OUR single-drug configs
#   auc               per-drug held-out test AUC of the same, + baseline columns
#   md_runs/md_meta   OUR joint (multi-drug) runs
#   MD_CMP[run]       per-drug frame lining a joint run up against its matched
#                     single-drug config AND the MD-CNN baseline
frames = []
for exp in EXPERIMENTS:
    s = pd.read_csv(EXP_DIR / exp / "summary.csv")
    s["drug"] = s["drug"].str.upper()
    s["experiment"], s["label"] = exp, LABEL[exp]
    frames.append(s)
long = pd.concat(frames, ignore_index=True)

our_cols = [LABEL[e] for e in EXPERIMENTS]
cv_mean = long.pivot(index="drug", columns="experiment", values="cv_auc_mean")[EXPERIMENTS]
cv_std  = long.pivot(index="drug", columns="experiment", values="cv_auc_std")[EXPERIMENTS]
auc     = long.pivot(index="drug", columns="experiment", values="test_auc")[EXPERIMENTS]
for _f in (cv_mean, cv_std, auc):
    _f.columns = [LABEL[c] for c in _f.columns]

base_col, leaky_col = "BIG-TB SD-CNN", "SD-CNN (pub, leaky)"
auc[base_col]  = pd.Series({d: v[0 if USE_CLEAN_BASELINE else 2]
                            for d, v in SDCNN_CLEAN.items()})
auc[leaky_col] = pd.Series({d: v[2] for d, v in SDCNN_CLEAN.items()})
auc = auc.loc[auc[our_cols].max(axis=1).sort_values(ascending=False).index]
cv_mean, cv_std = cv_mean.reindex(auc.index), cv_std.reindex(auc.index)

ALL_DRUGS = sorted(BIGTB_1DCNN)


def load_md_runs(exp_dir):
    """OUR joint runs. `cv_auc_std` (fold-to-fold SD) is recomputed from the run
    JSON — multidrug_summary.csv only carries the mean."""
    runs, meta = {}, {}
    for f in sorted(glob.glob(str(exp_dir / "multidrug_*" / "multidrug_summary.csv"))):
        run_dir = Path(f).parent
        df = pd.read_csv(f)
        df["drug"] = df["drug"].str.upper()
        df = df.set_index("drug")
        js = sorted(run_dir.glob("multidrug__*.json"))
        if js:
            raw = json.loads(js[0].read_text())
            df["cv_auc_std"] = pd.Series({
                d: (float(np.std(v, ddof=1)) if len(v) > 1 else np.nan)
                for d in raw.get("cv_per_drug_auc", {})
                for v in [[fo["per_drug"][d]["auc"] for fo in raw.get("cv_folds", [])
                           if d in fo.get("per_drug", {})]]})
            meta[run_dir.name] = raw
        runs[run_dir.name] = df
    return runs, meta


md_runs, md_meta = load_md_runs(EXP_DIR)
MD_ORDER = list(md_runs)
MD_COL = {r: MD_LABELS.get(r, r) for r in MD_ORDER}


def md_compare(run):
    """Per-drug: our joint run vs its matched single-drug config vs MD-CNN."""
    df = md_runs[run]
    sd = MD_SD_MATCH.get(run, LABEL[REFERENCE_EXP])
    sd = sd if sd in cv_mean.columns else LABEL[REFERENCE_EXP]
    ds = [d for d in ALL_DRUGS if d in df.index and d in BIGTB_MDCNN]
    t = pd.DataFrame(index=ds)
    t["joint cv"]     = df.loc[ds, "cv_auc"]
    t["joint cv sd"]  = df["cv_auc_std"].reindex(ds) if "cv_auc_std" in df else np.nan
    t["single cv"]    = cv_mean[sd].reindex(ds)
    t["single cv sd"] = cv_std[sd].reindex(ds)
    t["base cv"]      = [BIGTB_MDCNN[d][1] for d in ds]
    t["base cv sd"]   = [BIGTB_MDCNN[d][2] for d in ds]
    t["joint test"]   = df.loc[ds, "test_auc"]
    t["single test"]  = auc[sd].reindex(ds)
    t["base test"]    = [BIGTB_MDCNN[d][0] for d in ds]
    t["d vs single"]  = t["joint cv"] - t["single cv"]
    t["d vs base"]    = t["joint cv"] - t["base cv"]
    t["d vs base test"] = t["joint test"] - t["base test"]
    t.attrs["single_config"] = sd
    return t


MD_CMP = {r: md_compare(r) for r in MD_ORDER}

# ---- coverage: a blank cell in a table is an unfinished run, not a zero ----
_have = {e: set(long.loc[long.experiment == e, "drug"]) for e in EXPERIMENTS}
complete_drugs = [d for d in ALL_DRUGS if all(d in _have[e] for e in EXPERIMENTS)]
print(f"single-drug configs ({len(EXPERIMENTS)}):")
for e in EXPERIMENTS:
    miss = [d for d in ALL_DRUGS if d not in _have[e]]
    print(f"  {LABEL[e]:<20} {len(_have[e]):>2}/{len(ALL_DRUGS)} drugs"
          + ("" if not miss else "   MISSING: " + ", ".join(miss)))
print(f"joint runs ({len(md_runs)}):")
for r in MD_ORDER:
    raw = md_meta.get(r, {})
    print(f"  {MD_COL[r]:<20} {len(md_runs[r].drop(index=['MACRO'], errors='ignore')):>2}"
          f"/{len(ALL_DRUGS)} drugs   folder={r}  inputs={'+'.join(raw.get('modalities', [])) or '?'}")
if not md_runs:
    print("  none — launch: python run_multidrug.py --real --modalities dna --drugs all")
print(f"\ndrugs present in EVERY single-drug config: {len(complete_drugs)}/{len(ALL_DRUGS)}")

In [ ]:
# === TABLE 1 — why the SD-CNN baseline is corrected ========================
# Their published "test AUC" is the `assess` output, computed on a stratified
# re-split that overlaps the crossval TRAIN set (~80% of the "test" isolates were
# in training). `clean test` / `clean CV` re-evaluate THEIR saved best model on
# the truly held-out split — no retraining, and it reproduces their published
# number exactly for 10/11 drugs, which is what proves the reconstruction is
# faithful. Inflation lands on the hard imbalanced drugs — the ones we appeared
# to trail on. Everything downstream compares against `clean test` / `clean CV`.
_leak = pd.DataFrame({
    "published (leaky)": {d: v[2] for d, v in SDCNN_CLEAN.items()},
    "clean test":        {d: v[0] for d, v in SDCNN_CLEAN.items()},
    "clean CV":          {d: v[1] for d, v in SDCNN_CLEAN.items()}})
_leak["leak inflation"] = _leak["published (leaky)"] - _leak["clean test"]
_leak = _leak.sort_values("leak inflation", ascending=False)

display(_leak.style
        .format("{:.3f}", subset=["published (leaky)", "clean test", "clean CV"])
        .format("{:+.3f}", subset=["leak inflation"])
        .apply(lambda c: [BAD_BG if v > 0.02 else "" for v in c],
               subset=["leak inflation"])
        .set_table_styles(TABLE_STYLES)
        .set_caption(
            "<b>BIG-TB SD-CNN baseline correction.</b> Highlighted = the published "
            "number is materially inflated (>0.02) by the crossval/assess "
            "train-test leak — exactly the hard imbalanced drugs "
            "(MOXI / CAPREO / ETO). The MD-CNN baseline is NOT affected (its "
            "assess script splits on a stored cohort column, never re-splits)."))

In [ ]:
# === TABLE 2 — cross-validated AUC: every model, every drug (THE table) ====
# 5-fold mean ± SD. This is the metric to judge on: all four blocks below compute
# it the same way (non-stratified 80/20 + KFold-5 on the train split), so it is
# like-for-like across ours and theirs.
G_SD  = "OURS: single-drug  ·  one model per drug"
G_JT  = "OURS: joint  ·  one model, all 11 drugs"
G_BASE = "BASELINE: BIG-TB (published)"
C_BSD = "SD-CNN\n(single-drug)"
C_BMD = "MD-CNN\n(joint)"

_cols = {}
for c in our_cols:
    _cols[(G_SD, c)] = [pm(m, s) for m, s in zip(cv_mean[c], cv_std[c])]
for r in MD_ORDER:
    _d = md_runs[r]
    _cols[(G_JT, MD_COL[r].replace("OURS: ", ""))] = [
        pm(_d["cv_auc"].get(d, np.nan),
           _d["cv_auc_std"].get(d, np.nan) if "cv_auc_std" in _d else np.nan)
        for d in cv_mean.index]
_cols[(G_BASE, C_BSD)] = [pm(SDCNN_CLEAN.get(d, (np.nan,) * 3)[1],
                             SDCNN_CV_SD.get(d, np.nan)) for d in cv_mean.index]
_cols[(G_BASE, C_BMD)] = [
    pm(BIGTB_MDCNN.get(d, (np.nan,) * 3)[1], BIGTB_MDCNN.get(d, (np.nan,) * 3)[2])
    + (" ⚠" if d in MDCNN_CV_SUSPECT else "") for d in cv_mean.index]

cv_tab = pd.DataFrame(_cols, index=cv_mean.index)
# numeric twin, used for the MEAN row and for bolding our best per drug
_num = pd.DataFrame(
    {c: pd.to_numeric([str(v).split(" ")[0] for v in cv_tab[c]], errors="coerce")
     for c in cv_tab.columns}, index=cv_tab.index)
cv_tab.loc[f"MEAN over {len(cv_tab)} drugs"] = [
    f"{_num[c].mean():.3f}" if _num[c].notna().any() else "—" for c in cv_tab.columns]
_ours_mi = [c for c in cv_tab.columns if c[0] in (G_SD, G_JT)]

display(cv_tab.style
        .apply(lambda row: ["font-weight:bold"
                            if (row.name in _num.index and c in _ours_mi
                                and _num.loc[row.name, c] == _num.loc[row.name, _ours_mi].max())
                            else "" for c in cv_tab.columns], axis=1)
        .set_table_styles(TABLE_STYLES)
        .set_caption(
            "<b>Cross-validated AUC (5-fold mean ± SD) — the reliable metric.</b> "
            "Bold = our best model for that drug (single-drug or joint). "
            "A high SD (e.g. LEVOFLOXACIN, 269 isolates) means that drug's numbers "
            "are noise-dominated on <i>both</i> sides.<br>"
            "⚠ = the paper's Table-14 CV value for ETHIONAMIDE contradicts the "
            "authors' own auc.csv (0.865 vs 0.622) and its Table-4 test AUC "
            "(0.709); treat that one baseline cell as unreliable."))

In [ ]:
# === TABLE 3 — held-out test AUC, plus the two headline gaps ===============
# SINGLE 20% split of one best-CV-fold model — high variance. Shown because it is
# the metric BIG-TB publishes. The two Δ columns are the summary of this whole
# notebook: our best single-drug config vs SD-CNN, and our joint model vs MD-CNN.
C_DSD = "Δ  our best single-drug\n− SD-CNN"
C_DMD = "Δ  our joint\n− MD-CNN"
G_DELTA = "THE GAPS"

_cols = {(G_SD, c): auc[c] for c in our_cols}
for r in MD_ORDER:
    _cols[(G_JT, MD_COL[r].replace("OURS: ", ""))] = md_runs[r]["test_auc"].reindex(auc.index)
_cols[(G_BASE, C_BSD)] = auc[base_col]
_cols[(G_BASE, "SD-CNN\n(pub, leaky)")] = auc[leaky_col]
_cols[(G_BASE, C_BMD)] = pd.Series({d: v[0] for d, v in BIGTB_MDCNN.items()}).reindex(auc.index)
_cols[(G_DELTA, C_DSD)] = auc[our_cols].max(axis=1) - auc[base_col]
if MD_ORDER:
    _r0 = MD_ORDER[0]
    _cols[(G_DELTA, C_DMD)] = (md_runs[_r0]["test_auc"].reindex(auc.index)
                               - pd.Series({d: v[0] for d, v in BIGTB_MDCNN.items()}))

test_tab = pd.DataFrame(_cols, index=auc.index)
test_tab.loc[f"MEAN over {len(auc)} drugs"] = test_tab.mean()
_our_test_mi = [c for c in test_tab.columns if c[0] in (G_SD, G_JT)]
_delta_mi = [c for c in test_tab.columns if c[0] == G_DELTA]

display(test_tab.style
        .format("{:.3f}", na_rep="—")
        .format("{:+.3f}", subset=_delta_mi, na_rep="—")
        .background_gradient(subset=_our_test_mi, cmap="Blues", vmin=0.5, vmax=1.0, axis=None)
        .apply(lambda row: ["font-weight:bold"
                            if (c in _our_test_mi and row[c] == row[_our_test_mi].max())
                            else "" for c in test_tab.columns], axis=1)
        .apply(colour_delta, subset=_delta_mi)
        .set_table_styles(TABLE_STYLES)
        .set_caption(
            "<b>Held-out test AUC (single 20% split — high variance).</b> Use "
            "Table 2 for the reliable comparison. Bold = our best model per drug; "
            "— = not run yet. Green Δ = we're ahead.<br>"
            "Caveat on the joint-vs-MD-CNN Δ: their test cohort is the predefined "
            "'not set1_original_10202' set, ours is a random 20% hold-out — the CV "
            "columns in Table 2 are the matched comparison."))

In [ ]:
# === FIGURE 1 — our single-drug models vs the leak-corrected SD-CNN ========
# Same model class on both sides (one DNA CNN per drug), so this is the cleanest
# ours-vs-theirs read. Left = 5-fold CV (reliable), right = single split (noisy).
PLOT_SD_CONFIG = "DNA"          # which of OUR single-drug configs to plot

_drugs = [d for d in cv_mean.index if d in SDCNN_CLEAN
          and not pd.isna(cv_mean.loc[d, PLOT_SD_CONFIG])]
_order = sorted(_drugs, key=lambda d: SDCNN_CLEAN[d][1])      # hardest baseline top
_panels = [
    dict(title="Cross-validated AUC (5-fold mean)",
         sub="the reliable metric — both sides are 5-fold CV means",
         ours=cv_mean.loc[_order, PLOT_SD_CONFIG],
         ours_sd=cv_std.loc[_order, PLOT_SD_CONFIG],
         theirs=pd.Series({d: SDCNN_CLEAN[d][1] for d in _order}),
         theirs_sd=pd.Series({d: SDCNN_CV_SD.get(d, np.nan) for d in _order}),
         ghost=None),
    dict(title="Held-out test AUC (single 20% split)",
         sub="noisy on both sides — × marks the paper's leaky published value",
         ours=auc.loc[_order, PLOT_SD_CONFIG],
         ours_sd=pd.Series(np.nan, index=_order),
         theirs=pd.Series({d: SDCNN_CLEAN[d][0] for d in _order}),
         theirs_sd=pd.Series(np.nan, index=_order),
         ghost=pd.Series({d: SDCNN_CLEAN[d][2] for d in _order})),
]

with mpl.rc_context(RC):
    fig, axes = plt.subplots(1, 2, figsize=(14, 0.52 * len(_order) + 3.4), sharey=True)
    y = np.arange(len(_order))[::-1]
    for ax, P in zip(axes, _panels):
        a, b = P["theirs"], P["ours"]                       # baseline -> ours
        for yi, d in zip(y, _order):
            for v, sd in ((a[d], P["theirs_sd"][d]), (b[d], P["ours_sd"][d])):
                if not pd.isna(sd):                         # fold-noise band
                    ax.plot([v - sd, v + sd], [yi, yi], color="0.80", lw=6,
                            solid_capstyle="butt", zorder=1)
            c = BLUE if b[d] >= a[d] else VERM
            ax.annotate("", xy=(b[d], yi), xytext=(a[d], yi), zorder=3,
                        arrowprops=dict(arrowstyle="-|>", color=c, lw=2.4,
                                        shrinkA=0, shrinkB=0, mutation_scale=15))
        if P["ghost"] is not None:
            ax.scatter(P["ghost"].values, y, marker="x", s=55, color=GREY, lw=1.8, zorder=4)
        ax.scatter(a.values, y, s=95, facecolor="white", edgecolor=ORANGE, lw=2.4, zorder=5)
        ax.scatter(b.values, y, s=95, zorder=6, edgecolor="white", lw=1.2,
                   color=[BLUE if b[d] >= a[d] else VERM for d in _order])
        gut = blended_transform_factory(ax.transAxes, ax.transData)
        for yi, d in zip(y, _order):
            dv = b[d] - a[d]
            ax.text(0.995, yi, f"{dv:+.3f}", transform=gut, va="center", ha="right",
                    fontsize=10, fontweight="bold", color=BLUE if dv >= 0 else VERM)
        ax.text(0.995, 1.004, "Δ vs baseline", transform=ax.transAxes, va="bottom",
                ha="right", fontsize=9.5, color="0.45")
        lo = float(min(a.min(), b.min()))
        ax.set_xlim(lo - 0.04, 1.20)                        # right margin = Δ gutter
        ax.set_xticks([t for t in np.arange(0.6, 1.01, 0.1) if t >= lo - 0.04])
        ax.set_xlabel("AUC")
        ax.text(0, 1.085, P["title"], transform=ax.transAxes, fontsize=12.5,
                fontweight="bold", va="bottom")
        ax.text(0, 1.043, "we match or beat the corrected baseline on "
                f"{int((b >= a).sum())} of {len(_order)} drugs",
                transform=ax.transAxes, fontsize=10.5, color="0.2", va="bottom",
                fontweight="bold")
        ax.text(0, 1.004, P["sub"], transform=ax.transAxes, fontsize=10,
                color="0.45", va="bottom")
        ax.grid(axis="x", alpha=0.25)
        ax.set_axisbelow(True)
    axes[0].set_yticks(y, _order, fontsize=10)
    fig.legend(handles=[
        Line2D([], [], marker="o", ls="", mfc="white", mec=ORANGE, mew=2.4, ms=10,
               label="BIG-TB SD-CNN (leak-corrected)"),
        Line2D([], [], marker="o", ls="", color=BLUE, ms=10,
               label=f"ours, {PLOT_SD_CONFIG} single-drug — ahead"),
        Line2D([], [], marker="o", ls="", color=VERM, ms=10, label="ours — behind"),
        Line2D([], [], marker="x", ls="", color=GREY, mew=1.8, ms=8,
               label="their published (leaky) test value"),
        Line2D([], [], color="0.80", lw=6, label="±1 SD across CV folds"),
    ], loc="lower center", bbox_to_anchor=(0.5, -0.005), ncol=5, fontsize=10,
       handletextpad=0.6, columnspacing=1.8)
    fig.suptitle(f"Fig 1 · our {PLOT_SD_CONFIG}-only single-drug models vs the "
                 "leak-corrected BIG-TB SD-CNN", x=0.01, ha="left", fontsize=14,
                 fontweight="bold")
    fig.text(0.01, -0.045, "One DNA CNN per drug on both sides. Drugs ordered by "
             "baseline difficulty (hardest at top). Arrow runs baseline → ours.",
             fontsize=9.5, color="0.45")
    fig.tight_layout(rect=(0, 0.055, 1, 0.945))
save_fig(fig, "fig1_single_drug_vs_sdcnn")
plt.show()

In [ ]:
# === FIGURE 2 — does adding a modality help? (Δ CV-AUC vs DNA-only) ========
# On CV, not test: the single-split deltas are dominated by split noise. A cell
# is starred when the change exceeds the two configs' combined fold-to-fold SD.
ref = LABEL[REFERENCE_EXP]
gain_cols = [c for c in our_cols if c != ref]
delta = cv_mean[gain_cols].sub(cv_mean[ref], axis=0)
noise = np.sqrt(cv_std[gain_cols] ** 2 + (cv_std[ref] ** 2).values[:, None])

vmax = float(np.nanmax(np.abs(delta.values)))
with mpl.rc_context(RC):
    fig, ax = plt.subplots(figsize=(1.5 * len(gain_cols) + 3.5, 0.46 * len(delta) + 2.4))
    im = ax.imshow(delta.values, cmap="RdBu",
                   norm=TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax), aspect="auto")
    for i in range(delta.shape[0]):
        for j in range(delta.shape[1]):
            v = delta.values[i, j]
            if np.isnan(v):
                continue
            big = abs(v) > noise.values[i, j]
            ax.text(j, i, f"{v:+.3f}" + ("★" if big else ""), ha="center", va="center",
                    fontsize=9, fontweight="bold" if big else "normal",
                    color="white" if abs(v) > 0.6 * vmax else "0.15")
    ax.set_xticks(range(len(gain_cols)), gain_cols, rotation=18, ha="right")
    ax.set_yticks(range(len(delta)), delta.index, fontsize=10)
    ax.set_xticks(np.arange(-.5, len(gain_cols), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(delta), 1), minor=True)
    ax.grid(which="minor", color="white", lw=2)
    ax.tick_params(which="minor", length=0)
    ax.set_title(f"Fig 2 · Δ CV-AUC vs {ref}-only  —  blue = the added modality helps\n",
                 loc="left")
    ax.text(0, 1.01, "★ = change exceeds the two configs' combined fold-to-fold SD",
            transform=ax.transAxes, fontsize=10, color="0.45", va="bottom")
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cb.set_label(f"CV-AUC − {ref}")
    fig.tight_layout()
save_fig(fig, "fig2_modality_gains")
plt.show()

_summary = pd.DataFrame([{
    "config": c,
    "mean Δ CV vs DNA": delta[c].mean(),
    "median Δ": delta[c].median(),
    "drugs better": int((delta[c] > 0).sum()),
    "drugs worse": int((delta[c] < 0).sum()),
    "beyond fold noise": int((delta[c].abs() > noise[c]).sum()),
} for c in gain_cols]).set_index("config").sort_values("mean Δ CV vs DNA", ascending=False)
display(_summary.style
        .format({"mean Δ CV vs DNA": "{:+.4f}", "median Δ": "{:+.4f}"})
        .apply(colour_delta, subset=["mean Δ CV vs DNA"])
        .set_table_styles(TABLE_STYLES)
        .set_caption("Per-config summary of Fig 2 — averaged over the drugs that "
                     "config has run."))

---

## Multi-drug: our joint model vs. the published BIG-TB MD-CNN

Above, every row was a *single-drug* model. `run_multidrug.py` trains **one**
`MultiDrugNet` over all 11 drugs — the same setting as BIG-TB's **MD-CNN (OHE)**,
which is therefore its baseline (not SD-CNN).

| metric | their source | comparable? |
|---|---|---|
| **CV AUC** (5-fold ± SD) | Table 14 | **yes — matched.** Their `main_mdcnn_crossval.py` uses a non-stratified 80/20 `train_test_split(random_state=42)` + `KFold(5)` on the train split; `run_multidrug_cv` does exactly the same. |
| **test AUC** | Table 4 | indicative only. Their `main_mdcnn_assess.py` trains on the cohort `category == "set1_original_10202"` and tests on *every other* isolate — a different, much larger cohort than our random 20% hold-out. |

Because `assess` never calls `train_test_split`, the SD-CNN leak does **not**
apply to MD-CNN; its published numbers need no correction. One caveat: Table 14's
ETHIONAMIDE row (SD-CNN 0.865 / MD-CNN 0.916) contradicts the authors' own
`auc.csv` (0.622) and Table 4 (0.670 / 0.709), so that baseline cell is flagged ⚠
and reported separately in the headline below.

In [ ]:
# === FIGURE 3 — joint vs single-drug (ours only): does one model beat 11? ==
# Each joint run is compared against the single-drug config with the SAME inputs
# (DNA joint vs DNA single-drug), never against a per-drug max over configs.
if not md_runs:
    print("No joint runs yet.")
else:
    run = MD_ORDER[0]
    t = MD_CMP[run]
    sd_col = t.attrs["single_config"]
    order = t["single cv"].sort_values().index            # hardest drug on top
    with mpl.rc_context(RC):
        fig, ax = plt.subplots(figsize=(11, 0.52 * len(order) + 2.6))
        y = np.arange(len(order))[::-1]
        for yi, d in zip(y, order):
            a, b = t.loc[d, "single cv"], t.loc[d, "joint cv"]
            nz = np.sqrt(np.nan_to_num(t.loc[d, "single cv sd"]) ** 2
                         + np.nan_to_num(t.loc[d, "joint cv sd"]) ** 2)
            ax.plot([a - nz, a + nz], [yi, yi], color="0.85", lw=7,
                    solid_capstyle="butt", zorder=1)      # noise band around single
            c = BLUE if b >= a else VERM
            ax.annotate("", xy=(b, yi), xytext=(a, yi), zorder=3,
                        arrowprops=dict(arrowstyle="-|>", color=c, lw=2.6,
                                        shrinkA=0, shrinkB=0, mutation_scale=16))
            beyond = abs(b - a) > nz
            ax.text(max(a, b) + 0.008, yi, f"{b - a:+.3f}" + ("  ★" if beyond else ""),
                    va="center", ha="left", fontsize=10.5, fontweight="bold",
                    color=c)
        ax.scatter(t.loc[order, "single cv"], y, s=95, facecolor="white",
                   edgecolor=ORANGE, lw=2.4, zorder=4)
        ax.scatter(t.loc[order, "joint cv"], y, s=95, zorder=5, edgecolor="white",
                   lw=1.2, color=[BLUE if t.loc[d, "d vs single"] >= 0 else VERM
                                  for d in order])
        ax.set_yticks(y, list(order), fontsize=10)
        ax.set_xlim(float(min(t["single cv"].min(), t["joint cv"].min())) - 0.05, 1.06)
        ax.set_xlabel("Cross-validated AUC (5-fold mean)")
        n_up = int((t["d vs single"] > 0).sum())
        ax.set_title(f"Fig 3 · joint training moves {n_up} of {len(t)} drugs up\n",
                     loc="left")
        ax.text(0, 1.012, f"{MD_COL[run]} (1 model) vs single-drug '{sd_col}' "
                f"({len(t)} models) · same inputs · grey = ±1 SD combined fold noise, "
                "★ = change beyond it",
                transform=ax.transAxes, fontsize=10, color="0.45", va="bottom")
        ax.grid(axis="x", alpha=0.25)
        ax.set_axisbelow(True)
        fig.legend(handles=[
            Line2D([], [], marker="o", ls="", mfc="white", mec=ORANGE, mew=2.4,
                   ms=10, label=f"single-drug ({sd_col}), {len(t)} models"),
            Line2D([], [], marker="o", ls="", color=BLUE, ms=10,
                   label="joint model — improved"),
            Line2D([], [], marker="o", ls="", color=VERM, ms=10,
                   label="joint model — degraded"),
        ], loc="lower center", bbox_to_anchor=(0.5, -0.01), ncol=3, fontsize=10,
           handletextpad=0.6, columnspacing=2.0)
        fig.tight_layout(rect=(0, 0.06, 1, 1))
    save_fig(fig, "fig3_joint_vs_single")
    plt.show()

In [ ]:
# === FIGURE 4 — our joint model vs the published BIG-TB MD-CNN ============
if not md_runs:
    print("No joint runs yet.")
else:
    run = MD_ORDER[0]
    t = MD_CMP[run].sort_values("d vs base")
    y = np.arange(len(t))
    panels = [("d vs base", "joint cv", "base cv",
               f"Δ CV-AUC  ({MD_COL[run]} − MD-CNN)",
               "MATCHED protocol — judge on this panel"),
              ("d vs base test", "joint test", "base test",
               f"Δ test AUC  ({MD_COL[run]} − MD-CNN)",
               "different test cohorts — indicative only")]
    with mpl.rc_context(RC):
        fig, axes = plt.subplots(1, 2, figsize=(13.5, 0.52 * len(t) + 2.8), sharey=True)
        for ax, (dcol, ocol, bcol, xlabel, sub) in zip(axes, panels):
            v = t[dcol].values
            flagged = np.array([d in MDCNN_CV_SUSPECT and dcol == "d vs base"
                                for d in t.index])
            ax.barh(y, v, height=0.66, zorder=2,
                    color=[GREY if f else (BLUE if x >= 0 else VERM)
                           for x, f in zip(v, flagged)])
            ax.axvline(0, color="0.25", lw=1.2, zorder=3)
            m = float(np.nanmax(np.abs(v)))
            for yi, d in zip(y, t.index):
                x = t.loc[d, dcol]
                lbl = (f"{x:+.3f}  ({t.loc[d, ocol]:.3f} vs {t.loc[d, bcol]:.3f})"
                       + ("  ⚠" if (d in MDCNN_CV_SUSPECT and dcol == "d vs base") else ""))
                inside = abs(x) > 0.55 * m       # long bar: label the empty half
                ax.text((0.02 * m if x < 0 else -0.02 * m) if inside
                        else x + 0.02 * m * (1 if x >= 0 else -1), yi, lbl,
                        va="center", fontsize=9, color="0.2",
                        ha=("left" if x < 0 else "right") if inside
                           else ("left" if x >= 0 else "right"))
            ax.set_xlim(-1.95 * m, 1.95 * m)
            ax.set_xlabel(xlabel)
            ax.set_title(f"we are ahead on {int((t[dcol] > 0).sum())} of {len(t)} drugs\n",
                         loc="left")
            ax.text(0, 1.012, sub, transform=ax.transAxes, fontsize=10,
                    color="0.45", va="bottom")
            ax.grid(axis="x", alpha=0.25)
            ax.set_axisbelow(True)
        axes[0].set_yticks(y, list(t.index), fontsize=10)
        fig.suptitle(f"Fig 4 · {MD_COL[run]} vs the published BIG-TB MD-CNN (OHE)",
                     x=0.01, ha="left", fontsize=14, fontweight="bold")
        fig.tight_layout(rect=(0, 0, 1, 0.97))
    save_fig(fig, "fig4_joint_vs_mdcnn")
    plt.show()

In [ ]:
# === FIGURE 5 — everything at a glance: mean AUC per model family =========
# Same drug set in every bar. Dots are individual drugs, so the spread behind
# each mean stays visible.
drugs = list(cv_mean.index)
n_sd = len(drugs)
rows_cv = [("BIG-TB SD-CNN\n(leak-corrected)",
            pd.Series({d: SDCNN_CLEAN[d][1] for d in drugs if d in SDCNN_CLEAN}),
            f"{n_sd} models", GREY),
           ("BIG-TB MD-CNN\n(published)",
            pd.Series({d: BIGTB_MDCNN[d][1] for d in drugs if d in BIGTB_MDCNN}),
            "1 model", "0.55"),
           (f"ours: single-drug\n{LABEL[REFERENCE_EXP]}",
            cv_mean[LABEL[REFERENCE_EXP]], f"{n_sd} models", ORANGE)]
rows_ts = [("BIG-TB SD-CNN\n(leak-corrected)", auc[base_col], f"{n_sd} models", GREY),
           ("BIG-TB MD-CNN\n(published)",
            pd.Series({d: BIGTB_MDCNN[d][0] for d in drugs if d in BIGTB_MDCNN}),
            "1 model", "0.55"),
           (f"ours: single-drug\n{LABEL[REFERENCE_EXP]}",
            auc[LABEL[REFERENCE_EXP]], f"{n_sd} models", ORANGE)]
for r in MD_ORDER:
    lab = MD_COL[r].replace("OURS: ", "ours: ").replace(" (", "\n(")
    rows_cv.append((lab, md_runs[r]["cv_auc"].reindex(drugs), "1 model", BLUE))
    rows_ts.append((lab, md_runs[r]["test_auc"].reindex(drugs), "1 model", BLUE))
rows_cv.append((f"ours: best-of-{len(our_cols)}*", cv_mean[our_cols].max(axis=1),
                f"{n_sd * len(our_cols)} models", "0.78"))
rows_ts.append((f"ours: best-of-{len(our_cols)}*", auc[our_cols].max(axis=1),
                f"{n_sd * len(our_cols)} models", "0.78"))

with mpl.rc_context(RC):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6.2))
    for ax, rows, title, sub in zip(
            axes, (rows_cv, rows_ts),
            (f"Mean CV-AUC across {n_sd} drugs", f"Mean test AUC across {n_sd} drugs"),
            ("the reliable metric (5-fold mean)",
             "single 20% split — noisy, but what BIG-TB publishes")):
        xs = np.arange(len(rows))
        means = [s.dropna().mean() for _, s, _, _ in rows]
        ax.bar(xs, means, width=0.62, color=[c for *_, c in rows], zorder=2)
        rng = np.random.default_rng(0)
        for xi, (_, s, _, _) in zip(xs, rows):
            v = s.dropna().values
            ax.scatter(xi + rng.uniform(-0.16, 0.16, v.size), v, s=22, color="white",
                       edgecolor="0.25", lw=0.8, zorder=4, alpha=0.9)
        for xi, mv in zip(xs, means):
            ax.text(xi, mv + 0.006, f"{mv:.3f}", ha="center", va="bottom",
                    fontsize=11, fontweight="bold", zorder=6,
                    path_effects=[pe.withStroke(linewidth=3.5, foreground="white")])
        ax.set_xticks(xs, [f"{lab}\n{note}" for lab, _, note, _ in rows], fontsize=9.5)
        ax.set_ylim(0.60, 1.005)
        ax.set_ylabel("AUC")
        ax.set_title(title + "\n", loc="left")
        ax.text(0, 1.012, sub, transform=ax.transAxes, fontsize=10, color="0.45",
                va="bottom")
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)
    fig.suptitle("Fig 5 · every model family, one bar each", x=0.01, ha="left",
                 fontsize=14, fontweight="bold")
    fig.text(0.01, -0.02, f"* best-of-{len(our_cols)} is the per-drug maximum over "
             f"our {len(our_cols)} single-drug configs — a selection-biased ceiling, "
             "shown for reference only. Dots = individual drugs.",
             fontsize=9.5, color="0.45")
    fig.tight_layout(rect=(0, 0, 1, 0.95))
save_fig(fig, "fig5_summary")
plt.show()

In [ ]:
# === HEADLINE NUMBERS ======================================================
ref = LABEL[REFERENCE_EXP]
print("SINGLE-DRUG  (ours, one model per drug)  vs  BIG-TB SD-CNN, leak-corrected")
print("=" * 78)
_sd_base = pd.Series({d: SDCNN_CLEAN[d][1] for d in cv_mean.index if d in SDCNN_CLEAN})
for c in sorted(our_cols, key=lambda c: cv_mean[c].mean(), reverse=True):
    d = (cv_mean[c] - _sd_base).dropna()
    print(f"  {c:<20} CV {cv_mean[c].mean():.4f}  (baseline {_sd_base.mean():.4f})"
          f"   Δ {d.mean():+.4f}   {int((d > 0).sum())}/{len(d)} drugs ahead")
_gain = cv_mean[[c for c in our_cols if c != ref]].sub(cv_mean[ref], axis=0)
print(f"\n  adding modalities to {ref}: {int((_gain > 0).sum().sum())}/"
      f"{int(_gain.notna().sum().sum())} (drug, config) cells improve, "
      f"mean Δ {np.nanmean(_gain.values):+.4f}")

if md_runs:
    for run in MD_ORDER:
        t, sd_col = MD_CMP[run], MD_CMP[run].attrs["single_config"]
        ok = t.drop(index=[d for d in t.index if d in MDCNN_CV_SUSPECT])
        print(f"\n{MD_COL[run]}  (1 model, all drugs)   folder={run}")
        print("=" * 78)
        print(f"  vs OUR single-drug '{sd_col}' (11 models), CV:"
              f"   ours {t['joint cv'].mean():.4f}  single {t['single cv'].mean():.4f}"
              f"   Δ {t['d vs single'].mean():+.4f}"
              f"   {int((t['d vs single'] > 0).sum())}/{len(t)} drugs up")
        _corr = np.corrcoef(t['single cv'].rank(), t['d vs single'].rank())[0, 1]
        print(f"     gain vs difficulty: Spearman ρ = {_corr:+.2f}"
              "  (negative = joint training helps most where single-drug is weakest)")
        print(f"  vs BIG-TB MD-CNN, CV (matched):"
              f"    ours {t['joint cv'].mean():.4f}  MD-CNN {t['base cv'].mean():.4f}"
              f"   Δ {t['d vs base'].mean():+.4f}"
              f"   {int((t['d vs base'] > 0).sum())}/{len(t)} ahead")
        print(f"     excl. flagged {'/'.join(sorted(MDCNN_CV_SUSPECT))}:"
              f"       ours {ok['joint cv'].mean():.4f}  MD-CNN {ok['base cv'].mean():.4f}"
              f"   Δ {ok['d vs base'].mean():+.4f}"
              f"   {int((ok['d vs base'] > 0).sum())}/{len(ok)} ahead")
        print(f"  vs BIG-TB MD-CNN, test (different cohorts):"
              f" ours {t['joint test'].mean():.4f}  MD-CNN {t['base test'].mean():.4f}"
              f"   Δ {t['d vs base test'].mean():+.4f}"
              f"   {int((t['d vs base test'] > 0).sum())}/{len(t)} ahead")
        print(f"  biggest CV gap: {t['d vs base'].idxmin():<13} "
              f"{t['d vs base'].min():+.3f}   best: {t['d vs base'].idxmax():<13} "
              f"{t['d vs base'].max():+.3f}")

print("\nCaveats: CV is the matched, leak-free comparison everywhere; test columns "
      "use\ndifferent hold-out cohorts on the multi-drug side. LEVOFLOXACIN has 269 "
      "phenotyped\nisolates in total, so every LEVO number on either side is noise.")